# ASL Alphabet Recognition — ResNet50 (Preprocessing + Training + Validation)

This notebook implements the full pipeline for training a **ResNet50**-based ASL alphabet recognition model.

### Key differences from the MobileNetV2 pipeline
| Property | MobileNetV2 | ResNet50 |
|---|---|---|
| Input size | 128 × 128 | **224 × 224** |
| Color mode | Grayscale → 3ch replicated | **RGB (native)** |
| Normalization | `[-1, 1]` | **ImageNet mean/std** (`tf.keras.applications.resnet50.preprocess_input`) |
| Pretrained weights | ImageNet | ImageNet |

### Notebook structure
1. Imports & global config  
2. Class mapping  
3. ResNet50-compatible preprocessing  
4. Dataset loading & splitting  
5. `tf.data` pipeline  
6. Model construction (feature extraction → fine-tuning)  
7. Training (Phase 1 — frozen backbone)  
8. Fine-tuning (Phase 2 — partial unfreeze)  
9. Evaluation & visualisation  
10. Artefact export  

## 1 · Imports & Global Configuration

In [ ]:
import os
import json
import math
import random
from pathlib import Path
from typing import Tuple, List

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.utils import to_categorical

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ── ResNet50 Image Requirements ───────────────────────────────────────────────
# ResNet50 expects 224×224 RGB images normalised via resnet50.preprocess_input
# which converts [0,255] uint8 → ImageNet channel-wise mean subtraction (BGR order internally).
IMAGE_SIZE   = 224          # ResNet50 native resolution
NUM_CHANNELS = 3            # RGB — no grayscale replication needed
RANDOM_STATE = SEED

# ── Training Hyperparameters ─────────────────────────────────────────────────
BATCH_SIZE      = 32
EPOCHS_PHASE1   = 15        # frozen backbone
EPOCHS_PHASE2   = 20        # fine-tune top layers
LR_PHASE1       = 1e-3
LR_PHASE2       = 1e-5      # low LR to avoid destroying pretrained weights
UNFREEZE_FROM   = 143       # unfreeze ResNet50 layers from this index onward
DROPOUT_RATE    = 0.4
L2_REG          = 1e-4

# ── Paths ────────────────────────────────────────────────────────────────────
RAW_DATA_DIR       = Path("../data/raw/train")
PROCESSED_DATA_DIR = Path("../data/processed_resnet")
MODEL_DIR          = Path("../models")
REPORTS_DIR        = Path("../reports")

for d in [PROCESSED_DATA_DIR, MODEL_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories ready.")

## 2 · Class Mapping

In [ ]:
class_names = sorted([
    d.name for d in RAW_DATA_DIR.iterdir()
    if d.is_dir()
])

num_classes     = len(class_names)
class_to_index  = {cls: idx for idx, cls in enumerate(class_names)}
index_to_class  = {idx: cls for cls, idx in class_to_index.items()}

print(f"Detected {num_classes} classes:")
print(class_names)

# Persist mappings
with open(PROCESSED_DATA_DIR / "class_mapping.json", "w") as f:
    json.dump({
        "class_to_index" : class_to_index,
        "index_to_class" : {str(k): v for k, v in index_to_class.items()}
    }, f, indent=4)
print("Class mapping saved.")

## 3 · ResNet50-Compatible Preprocessing

Unlike MobileNetV2 (which used `[-1, 1]` normalisation on grayscale images),  
ResNet50 expects:
- **RGB** images at **224 × 224**
- Pixel values passed through `tensorflow.keras.applications.resnet50.preprocess_input`  
  (converts `[0, 255]` → channel-wise ImageNet mean subtraction, **no division by 255** beforehand)

In [ ]:
def preprocess_image(image_path: Path) -> np.ndarray:
    """
    Load and preprocess a single image for ResNet50.

    Steps
    -----
    1. Load as BGR (OpenCV default)
    2. Convert BGR → RGB  (ResNet50 trained on RGB)
    3. Resize to 224×224  (bilinear)
    4. Cast to float32 — keep pixel range [0, 255]
    5. Apply resnet50.preprocess_input  (ImageNet mean subtraction)

    Returns
    -------
    np.ndarray of shape (224, 224, 3), dtype float32
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Failed to read image: {image_path}")

    # BGR → RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Resize
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)

    # float32 in [0, 255] — preprocess_input expects this range
    img = img.astype(np.float32)

    # ResNet50 ImageNet normalisation
    img = resnet_preprocess(img)

    return img

### Preprocessing sanity-check

In [ ]:
# Visual sanity-check on one sample per class (shows first 10 classes)
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

for ax, cls in zip(axes, class_names[:10]):
    sample = next((RAW_DATA_DIR / cls).iterdir())
    img_raw = cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2RGB)
    ax.imshow(cv2.resize(img_raw, (IMAGE_SIZE, IMAGE_SIZE)))
    ax.set_title(cls, fontsize=12)
    ax.axis("off")

plt.suptitle("Sample images (RGB, pre-normalisation)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4 · Dataset Loading, Splitting & Saving

In [ ]:
X: List[np.ndarray] = []
y: List[int] = []

for class_name in class_names:
    class_dir = RAW_DATA_DIR / class_name
    label = class_to_index[class_name]

    for img_file in class_dir.iterdir():
        if img_file.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        try:
            img = preprocess_image(img_file)
            X.append(img)
            y.append(label)
        except Exception as e:
            print(f"Skipping {img_file}: {e}")

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print(f"Dataset loaded — X: {X.shape}, y: {y.shape}")
print(f"Pixel range after preprocess_input: [{X.min():.2f}, {X.max():.2f}]")

In [ ]:
# ── Stratified 80/20 split ────────────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

y_train_oh = to_categorical(y_train, num_classes)
y_val_oh   = to_categorical(y_val,   num_classes)

print("Training   :", X_train.shape, y_train_oh.shape)
print("Validation :", X_val.shape,   y_val_oh.shape)

In [ ]:
# ── Persist preprocessed arrays ───────────────────────────────────────────────
np.save(PROCESSED_DATA_DIR / "X_train.npy", X_train)
np.save(PROCESSED_DATA_DIR / "X_val.npy",   X_val)
np.save(PROCESSED_DATA_DIR / "y_train.npy", y_train_oh)
np.save(PROCESSED_DATA_DIR / "y_val.npy",   y_val_oh)
print("Preprocessed arrays saved to", PROCESSED_DATA_DIR)

## 5 · `tf.data` Input Pipeline

Using `tf.data` for efficient batching, prefetching, and on-the-fly augmentation during training.

In [ ]:
# ── Augmentation layer (training only) ───────────────────────────────────────
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),           # ±~29°
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name="augmentation")

AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(X: np.ndarray, y: np.ndarray,
                 batch_size: int, augment: bool = False) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if augment:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED)
        ds = ds.map(
            lambda x, lbl: (data_augmentation(x, training=True), lbl),
            num_parallel_calls=AUTOTUNE
        )
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(X_train, y_train_oh, BATCH_SIZE, augment=True)
val_ds   = make_dataset(X_val,   y_val_oh,   BATCH_SIZE, augment=False)

print(f"Train batches : {len(train_ds)}")
print(f"Val batches   : {len(val_ds)}")

## 6 · Model Construction

Strategy:
- Load `ResNet50(weights='imagenet', include_top=False)` as the backbone
- Freeze all backbone layers for **Phase 1** (train only the head)
- Unfreeze the top ResNet50 block for **Phase 2** (fine-tuning)

In [ ]:
def build_model(num_classes: int,
                dropout_rate: float = DROPOUT_RATE,
                l2_reg: float = L2_REG) -> keras.Model:
    """
    ResNet50 transfer-learning model for ASL classification.

    Architecture
    ------------
    Input (224, 224, 3)
      └─ ResNet50 backbone (frozen initially)
         └─ GlobalAveragePooling2D
            └─ BatchNormalization
               └─ Dense(512, relu, L2)
                  └─ Dropout
                     └─ Dense(256, relu, L2)
                        └─ Dropout
                           └─ Dense(num_classes, softmax)
    """
    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, NUM_CHANNELS), name="input")

    backbone = ResNet50(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs
    )
    backbone.trainable = False   # freeze for Phase 1

    x = backbone.output
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="bn_head")(x)

    x = layers.Dense(512, activation="relu",
                     kernel_regularizer=regularizers.l2(l2_reg),
                     name="dense_512")(x)
    x = layers.Dropout(dropout_rate, name="drop_1")(x)

    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=regularizers.l2(l2_reg),
                     name="dense_256")(x)
    x = layers.Dropout(dropout_rate, name="drop_2")(x)

    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = keras.Model(inputs, outputs, name="ASL_ResNet50")
    return model, backbone

model, backbone = build_model(num_classes)
model.summary(show_trainable=True)

## 7 · Phase 1 — Feature Extraction (Frozen Backbone)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_PHASE1),
    loss="categorical_crossentropy",
    metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")]
)

cb_phase1 = [
    callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / "resnet50_asl_phase1_best.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.TensorBoard(
        log_dir=str(REPORTS_DIR / "tb_logs" / "phase1"),
        histogram_freq=1
    )
]

print(f"Phase 1: training head only — {EPOCHS_PHASE1} epochs max")
history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=cb_phase1,
    verbose=1
)

### Phase 1 Learning Curves

In [ ]:
def plot_history(history, title=""):
    h = history.history
    epochs = range(1, len(h["loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    axes[0].plot(epochs, h["loss"],     label="Train Loss",  linewidth=2)
    axes[0].plot(epochs, h["val_loss"], label="Val Loss",    linewidth=2, linestyle="--")
    axes[0].set_title(f"{title} — Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Accuracy
    axes[1].plot(epochs, h["accuracy"],     label="Train Acc",  linewidth=2)
    axes[1].plot(epochs, h["val_accuracy"], label="Val Acc",    linewidth=2, linestyle="--")
    if "top3_acc" in h:
        axes[1].plot(epochs, h["val_top3_acc"], label="Val Top-3 Acc",
                     linewidth=2, linestyle=":", color="green")
    axes[1].set_title(f"{title} — Accuracy")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(REPORTS_DIR / f"curves_{title.lower().replace(' ', '_')}.png", dpi=150)
    plt.show()

plot_history(history_p1, "Phase 1 — Feature Extraction")

## 8 · Phase 2 — Fine-Tuning (Partial Backbone Unfreeze)

We unfreeze ResNet50 layers from `conv5_block1` onward (index ≥ 143) and train with a very low learning rate to refine high-level features without destroying earlier representations.

In [ ]:
# ── Unfreeze top ResNet50 block ───────────────────────────────────────────────
backbone.trainable = True

for layer in backbone.layers[:UNFREEZE_FROM]:
    layer.trainable = False

# Keep BatchNorm layers frozen to avoid shifting running statistics
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for l in backbone.layers if l.trainable)
print(f"Trainable backbone layers: {trainable_count} / {len(backbone.layers)}")

# Recompile with low LR
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_PHASE2),
    loss="categorical_crossentropy",
    metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")]
)

cb_phase2 = [
    callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / "resnet50_asl_phase2_best.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-8,
        verbose=1
    ),
    callbacks.TensorBoard(
        log_dir=str(REPORTS_DIR / "tb_logs" / "phase2"),
        histogram_freq=1
    )
]

print(f"Phase 2: fine-tuning — {EPOCHS_PHASE2} epochs max")
history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    callbacks=cb_phase2,
    verbose=1
)

In [ ]:
plot_history(history_p2, "Phase 2 — Fine-Tuning")

## 9 · Evaluation & Visualisation

In [ ]:
# ── Load best Phase 2 checkpoint ──────────────────────────────────────────────
best_model = keras.models.load_model(MODEL_DIR / "resnet50_asl_phase2_best.keras")

val_loss, val_acc, val_top3 = best_model.evaluate(val_ds, verbose=0)
print(f"Validation Loss     : {val_loss:.4f}")
print(f"Validation Accuracy : {val_acc*100:.2f}%")
print(f"Validation Top-3 Acc: {val_top3*100:.2f}%")

In [ ]:
# ── Predictions on validation set ────────────────────────────────────────────
y_pred_probs = best_model.predict(val_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_val_oh, axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm_norm,
    annot=True, fmt=".2f",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues", linewidths=0.3,
    ax=ax, annot_kws={"size": 7}
)
ax.set_title("Normalised Confusion Matrix — ResNet50 ASL", fontsize=14, pad=14)
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_ylabel("True Label", fontsize=11)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "confusion_matrix_resnet50.png", dpi=150)
plt.show()

In [ ]:
# ── Per-class accuracy bar chart ─────────────────────────────────────────────
per_class_acc = cm_norm.diagonal()
sorted_idx = np.argsort(per_class_acc)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(
    [class_names[i] for i in sorted_idx],
    per_class_acc[sorted_idx],
    color=plt.cm.RdYlGn(per_class_acc[sorted_idx])
)
ax.set_xlabel("Accuracy", fontsize=11)
ax.set_title("Per-class Validation Accuracy", fontsize=13)
ax.axvline(x=per_class_acc.mean(), color="navy", linestyle="--",
           label=f"Mean: {per_class_acc.mean():.3f}")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "per_class_accuracy_resnet50.png", dpi=150)
plt.show()

In [ ]:
# ── Prediction samples grid ───────────────────────────────────────────────────
# Denormalise for display (reverse ImageNet mean subtraction approximately)
IMAGENET_MEAN = np.array([103.939, 116.779, 123.68], dtype=np.float32)  # BGR order from preprocess_input

def denormalize_resnet(img: np.ndarray) -> np.ndarray:
    """Reverse resnet50.preprocess_input for display (approximate)."""
    img = img + IMAGENET_MEAN[::-1]   # add RGB means
    img = np.clip(img, 0, 255).astype(np.uint8)
    return img

num_samples = 20
indices = np.random.choice(len(X_val), num_samples, replace=False)

fig, axes = plt.subplots(4, 5, figsize=(18, 14))
axes = axes.flatten()

for ax, idx in zip(axes, indices):
    img_disp = denormalize_resnet(X_val[idx])
    true_lbl = index_to_class[y_val[idx]]
    pred_lbl = index_to_class[y_pred[np.where(
        np.arange(len(X_val)) == idx
    )[0][0] if len(np.where(np.arange(len(X_val)) == idx)[0]) else 0]]

    pred_idx = y_pred[idx]
    pred_lbl = index_to_class[pred_idx]
    correct  = (true_lbl == pred_lbl)

    ax.imshow(img_disp)
    color = "green" if correct else "red"
    ax.set_title(f"True: {true_lbl}\nPred: {pred_lbl}",
                 color=color, fontsize=9, fontweight="bold")
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(3)

plt.suptitle("Prediction Samples (green=correct, red=wrong)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "prediction_samples_resnet50.png", dpi=150)
plt.show()

## 10 · Artefact Export

In [ ]:
# ── Save final model ──────────────────────────────────────────────────────────
final_model_path = MODEL_DIR / "resnet50_asl_final.keras"
best_model.save(final_model_path)
print(f"Final model saved: {final_model_path}")

# TFLite export for on-device / real-time inference
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]      # dynamic-range quantisation
tflite_model = converter.convert()

tflite_path = MODEL_DIR / "resnet50_asl.tflite"
tflite_path.write_bytes(tflite_model)
print(f"TFLite model saved : {tflite_path}")
print(f"TFLite size        : {tflite_path.stat().st_size / 1e6:.1f} MB")

In [ ]:
# ── Merge and save complete training history ──────────────────────────────────
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history.get(key, [])
    return merged

full_history = merge_histories(history_p1, history_p2)

with open(REPORTS_DIR / "training_history_resnet50.json", "w") as f:
    json.dump({
        k: [float(v) for v in vals]
        for k, vals in full_history.items()
    }, f, indent=4)
print("Training history saved.")

In [ ]:
# ── Training Summary ──────────────────────────────────────────────────────────
summary = {
    "model": "ResNet50",
    "pretrained_weights": "ImageNet",
    "num_classes": num_classes,
    "class_names": class_names,
    "image_preprocessing": {
        "target_size": [IMAGE_SIZE, IMAGE_SIZE],
        "color_mode": "RGB",
        "channels": NUM_CHANNELS,
        "normalization": "resnet50.preprocess_input (ImageNet mean subtraction)"
    },
    "data_split": {
        "train": int(len(X_train)),
        "validation": int(len(X_val)),
        "stratified": True,
        "random_state": RANDOM_STATE
    },
    "training": {
        "phase1_epochs_run": len(history_p1.history["loss"]),
        "phase1_lr": LR_PHASE1,
        "phase2_epochs_run": len(history_p2.history["loss"]),
        "phase2_lr": LR_PHASE2,
        "batch_size": BATCH_SIZE,
        "unfreeze_from_layer": UNFREEZE_FROM
    },
    "final_validation": {
        "loss": round(float(val_loss), 4),
        "accuracy": round(float(val_acc), 4),
        "top3_accuracy": round(float(val_top3), 4)
    },
    "artifacts": {
        "model_keras": str(final_model_path),
        "model_tflite": str(tflite_path),
        "class_mapping": str(PROCESSED_DATA_DIR / "class_mapping.json")
    }
}

with open(REPORTS_DIR / "training_summary_resnet50.json", "w") as f:
    json.dump(summary, f, indent=4)

print("\n" + "="*55)
print("  ASL ResNet50 Training Complete")
print("="*55)
print(f"  Val Accuracy : {val_acc*100:.2f}%")
print(f"  Val Top-3    : {val_top3*100:.2f}%")
print(f"  Val Loss     : {val_loss:.4f}")
print("="*55)